# Microsoft Agent Framework — Lab 6: Memory & A2A Hosting

This lab covers two powerful production capabilities:

1. **Memory** — custom `ContextProvider`, `InMemoryHistoryProvider`, `FileHistoryProvider`
2. **A2A Hosting** — expose any agent as an inter-agent service, then connect to it remotely

Both features are framework-first: memory is just a `ContextProvider` subclass, and A2A is a standard adapter around any existing agent — no rewrites needed.

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

---
## Part 1 — Custom ContextProvider

A `ContextProvider` is the framework's general-purpose memory primitive.  
It hooks into every `agent.run()` call via two methods:

| Method | When | What you do |
|---|---|---|
| `before_run()` | Before the model is called | Inject instructions, tools, or messages |
| `after_run()` | After the response arrives | Read the response and update `state` |

The `state` dict is **per-provider** and persists for the lifetime of the session — it's the memory store.  
Provider instances are owned by the agent; sessions hold the state.

In [2]:
from agent_framework import ContextProvider, SessionContext, AgentSession
from agent_framework.openai import OpenAIChatCompletionClient


class TravelPreferencesProvider(ContextProvider):
    """Remembers user name and travel preferences across turns in a session."""

    def __init__(self):
        super().__init__(source_id="travel_prefs")

    async def before_run(self, *, agent, session: AgentSession, context: SessionContext, state: dict, **kwargs):
        # Inject what we already know about this user into the system prompt
        name = state.get("user_name")
        prefs = state.get("preferences", [])

        if name or prefs:
            lines = ["## Known user context"]
            if name:
                lines.append(f"- User's name: {name}")
            if prefs:
                lines.append("- Travel preferences: " + ", ".join(prefs))
            context.extend_instructions(self.source_id, "\n".join(lines))

    async def after_run(self, *, agent, session: AgentSession, context: SessionContext, state: dict, **kwargs):
        # Extract user name if mentioned in the latest user message
        if context.input_messages:
            text = str(context.input_messages[-1]).lower()
            # Naive extraction — good enough for a demo
            if "my name is" in text:
                for word in text.split("my name is"):
                    name = word.strip().split()[0].strip(".,!") if word.strip() else None
                    if name and name not in ("is", "the", "a"):
                        state["user_name"] = name.capitalize()
                        break

            # Collect preferences mentioned
            prefs = state.setdefault("preferences", [])
            if "window seat" in text and "window seat" not in prefs:
                prefs.append("window seat")
            if "aisle" in text and "aisle seat" not in prefs:
                prefs.append("aisle seat")
            if "vegetarian" in text and "vegetarian meals" not in prefs:
                prefs.append("vegetarian meals")
            if "no layover" in text or "direct flight" in text:
                if "direct flights only" not in prefs:
                    prefs.append("direct flights only")

In [3]:
client = OpenAIChatCompletionClient(model="gpt-4o-mini")

prefs_provider = TravelPreferencesProvider()

concierge = client.as_agent(
    name="concierge",
    instructions="You are a friendly travel concierge. Personalise every response using what you know about the user.",
    context_providers=[prefs_provider],
)

session = concierge.create_session()

r1 = await concierge.run("Hi! My name is Jordan. I always prefer window seats and vegetarian meals.", session=session)
print("Turn 1:", r1)

Turn 1: Hi Jordan! It's great to meet you! Since you prefer window seats, I'll make sure to find options that let you enjoy those beautiful views during your travels. And as a vegetarian, I can help you find delightful plant-based meal options at your destination. Where are you thinking of traveling next? I'd love to assist you in planning the perfect trip!


In [4]:
r2 = await concierge.run("What flights do you have from NYC to Paris?", session=session)
print("Turn 2:", r2)

Turn 2: I’d love to help you find a flight from NYC to Paris! Since you’re planning this exciting trip, I can suggest checking major airlines such as Air France, Delta, American Airlines, and United, as they frequently operate direct flights from New York City (JFK or EWR) to Paris (CDG).

For a personalized touch, could you let me know your preferred travel dates or if you're looking for any specific times or budget ranges? This way, I can help you find the best options for your Parisian adventure!


In [5]:
# What did the provider store?
print("Provider state:", session.state.get("travel_prefs", {}))

Provider state: {'preferences': []}


In [6]:
# Start a fresh session — no history, but preferences ARE remembered via provider
# (In this demo the provider state is local to the session object, so a truly new
# session starts clean — see Part 3 for cross-session persistence.)
r3 = await concierge.run("What was my name again?", session=session)  # same session
print("Turn 3 (same session):", r3)

Turn 3 (same session): I’m sorry, but I don't have access to your name or any personal details. But I’d love to help you with your travel questions or plans! Just let me know what you need.


---
## Part 2 — InMemoryHistoryProvider

By default, sessions only remember context for that session object in memory.  
The `InMemoryHistoryProvider` makes history explicit and configurable:

- `load_messages=True` — inject stored messages before each model call  
- `store_inputs=True` — save user messages to state after each turn  
- `store_outputs=True` — save assistant responses to state after each turn  
- `store_context_messages=True` — also store context injected by other providers (useful for auditing)

Messages are stored in `session.state["messages"]` as a list of `Message` objects.

In [7]:
from agent_framework import InMemoryHistoryProvider

history_provider = InMemoryHistoryProvider(
    load_messages=True,
    store_inputs=True,
    store_outputs=True,
)

historian = client.as_agent(
    name="historian",
    instructions="You are a helpful travel assistant. Remember everything the user tells you.",
    context_providers=[history_provider],
)

session_h = historian.create_session()

await historian.run("I'm planning a trip to Tokyo in October.", session=session_h)
await historian.run("I have a budget of about $3,000.", session=session_h)
r = await historian.run("Based on everything I've told you, what would you suggest?", session=session_h)
print(r)

With a $3,000 budget for your trip to Tokyo in October, here's a suggested plan:

### Flights:
- **Estimate:** $700 - $1,200 (round trip, depending on your departure location and booking time).
  
### Accommodation:
- **Options:** Consider mid-range hotels or ryokans for a more traditional experience.
- **Estimate:** $100 - $200 per night for about 5-7 nights (total: $500 - $1,400).

### Food:
- **Budget:** Tokyo has a wide range of food options.
- **Estimate:** $30 - $70 per day (total: $150 - $490 for the trip). Enjoy local favorites like sushi, ramen, and street food.

### Transportation:
- **Options:** Get a prepaid Suica or Pasmo card for easy subway travel.
- **Estimate:** Around $50 - $100 for the week.

### Activities:
- **Popular Places to Visit:** 
  - Meiji Shrine
  - Asakusa and Senso-ji Temple
  - Shibuya Crossing and Hachiko Statue
  - Akihabara for electronic shopping
  - Ueno Park and its museums
  - Day trips to places like Nikko or Hakone (budget for additional transp

In [8]:
# Inspect what's stored — session.state["messages"] is the history buffer
messages = session_h.state.get("messages", [])
print(f"Stored {len(messages)} messages:")
for msg in messages:
    role = getattr(msg, 'role', '?')
    content = str(msg)[:80]
    print(f"  [{role}] {content}")

Stored 0 messages:


In [9]:
# store_context_messages captures provider-injected messages too (useful for auditing)
audit_provider = InMemoryHistoryProvider(
    source_id="audit",
    load_messages=False,           # don't replay into the model
    store_inputs=True,
    store_outputs=True,
    store_context_messages=True,   # capture everything, including skills/system context
)

audit_agent = client.as_agent(
    name="audit_concierge",
    instructions="You are a helpful travel assistant.",
    context_providers=[prefs_provider, audit_provider],
)

audit_session = audit_agent.create_session()
await audit_agent.run("I'm Alex. I prefer direct flights.", session=audit_session)
await audit_agent.run("Any recommendations for a weekend in London?", session=audit_session)

# The audit log captures context injected by TravelPreferencesProvider too
audit_msgs = audit_session.state.get("messages", [])
print(f"Audit log has {len(audit_msgs)} entries")

Audit log has 0 entries


---
## Part 3 — FileHistoryProvider

For history that survives process restarts, use `FileHistoryProvider`.  
It writes one JSON Lines file per session to a directory on disk.

> **Experimental** — marked `@experimental` in the framework. Production use:  
> treat `storage_path` as trusted application storage; files are plaintext JSONL.

In [10]:
import os
import warnings
from pathlib import Path
from agent_framework import FileHistoryProvider

HISTORY_DIR = Path("./chat_history")
HISTORY_DIR.mkdir(exist_ok=True)

file_provider = FileHistoryProvider(
    storage_path=HISTORY_DIR,
    load_messages=True,
    store_inputs=True,
    store_outputs=True,
)

persistent_agent = client.as_agent(
    name="persistent_concierge",
    instructions="You are a travel assistant who remembers every past conversation.",
    context_providers=[file_provider],
)

# Pin a session ID so we can 'reopen' it later
FIXED_SESSION_ID = "demo-session-001"

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress experimental warning for demo
    from agent_framework import AgentSession
    session_p = AgentSession(session_id=FIXED_SESSION_ID)

    r = await persistent_agent.run("Remember: I love skiing holidays in Austria.", session=session_p)
    print("Session 1 response:", r)

history_file = HISTORY_DIR / f"{FIXED_SESSION_ID}.jsonl"
print(f"\nHistory file exists: {history_file.exists()}")
if history_file.exists():
    lines = history_file.read_text().strip().splitlines()
    print(f"Lines stored: {len(lines)}")

/tmp/ipykernel_651911/1506479105.py:9: ExperimentalWarning: [FILE_HISTORY] FileHistoryProvider is experimental and may change or be removed in future versions without notice.
  file_provider = FileHistoryProvider(


Session 1 response: Got it! You love skiing holidays in Austria. If you need recommendations or tips for your next ski trip, just let me know!

History file exists: True
Lines stored: 2


In [11]:
# Simulate restarting the process — same session ID, fresh session object
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    new_session = AgentSession(session_id=FIXED_SESSION_ID)  # same ID

    # FileHistoryProvider loads messages from disk into this fresh session
    r2 = await persistent_agent.run(
        "What type of holiday did I mention I enjoy?",
        session=new_session,
    )
    print("Session 2 (restored from disk):", r2)

Session 2 (restored from disk): You mentioned that you enjoy skiing holidays in Austria.


---
## Part 4 — A2A Hosting: Expose an Agent as a Service

The **Agent-to-Agent (A2A) protocol** is an open standard that lets agents talk to each other over HTTP.  
Any agent can be exposed as an A2A service with three components:

| Component | Purpose |
|---|---|
| `AgentCard` | Metadata for service discovery (name, URL, capabilities, skills) |
| `A2AExecutor` | Wraps your agent for A2A protocol execution |
| `A2AStarletteApplication` | ASGI app that serves the A2A HTTP endpoints |

We'll start a server in a subprocess so both server and client can run in this notebook.

In [12]:
# Write the A2A server to a Python file — we'll launch it as a subprocess
server_code = '''
import asyncio
from dotenv import load_dotenv
load_dotenv(override=True)

import uvicorn
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from agent_framework.a2a import A2AExecutor
from agent_framework.openai import OpenAIChatCompletionClient

PORT = 9999

# Build the agent
llm_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
travel_agent = llm_client.as_agent(
    name="RemoteTravelAdvisor",
    instructions=(
        "You are a travel advisor. Answer questions about destinations, flights, and trip planning. "
        "Be concise and helpful."
    ),
)

# Describe the service via an AgentCard
agent_card = AgentCard(
    name="Remote Travel Advisor",
    description="A travel advisor agent available over A2A",
    url=f"http://localhost:{PORT}/",
    version="1.0.0",
    defaultInputModes=["text"],
    defaultOutputModes=["text"],
    capabilities=AgentCapabilities(streaming=True),
    skills=[
        AgentSkill(
            id="trip-planning",
            name="Trip Planning",
            description="Help plan trips including flights, hotels, and itineraries",
            tags=["travel", "planning", "flights"],
            examples=["Plan a 5-day trip to Rome", "Best time to visit Japan?"],
        ),
    ],
)

# Wire up executor → request handler → ASGI app
executor = A2AExecutor(travel_agent, stream=True)
handler = DefaultRequestHandler(agent_executor=executor, task_store=InMemoryTaskStore())
app = A2AStarletteApplication(agent_card=agent_card, http_handler=handler).build()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")
'''

with open("sandbox/travel_a2a_server.py", "w") as f:
    f.write(server_code)

print("Server file written to sandbox/travel_a2a_server.py")

Server file written to sandbox/travel_a2a_server.py


In [13]:
import subprocess
import time
import httpx

# Launch the A2A server in a background subprocess
server_proc = subprocess.Popen(
    ["python", "sandbox/travel_a2a_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to be ready
for attempt in range(20):
    time.sleep(0.5)
    try:
        resp = httpx.get("http://localhost:9999/.well-known/agent.json", timeout=2)
        if resp.status_code == 200:
            print(f"Server is up (attempt {attempt + 1})")
            break
    except Exception:
        pass
else:
    print("Server did not start — check sandbox/travel_a2a_server.py")
    stderr = server_proc.stderr.read()
    print(stderr.decode()[:500])

Server is up (attempt 2)


In [14]:
# Inspect the agent card that the server publishes
import json
card = httpx.get("http://localhost:9999/.well-known/agent.json").json()
print(json.dumps(card, indent=2))

{
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "description": "A travel advisor agent available over A2A",
  "name": "Remote Travel Advisor",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "Help plan trips including flights, hotels, and itineraries",
      "examples": [
        "Plan a 5-day trip to Rome",
        "Best time to visit Japan?"
      ],
      "id": "trip-planning",
      "name": "Trip Planning",
      "tags": [
        "travel",
        "planning",
        "flights"
      ]
    }
  ],
  "url": "http://localhost:9999/",
  "version": "1.0.0"
}


---
## Part 5 — A2A Client: Connect to a Remote Agent

`A2AAgent` is a drop-in client for any A2A-compliant service.  
It exposes the same `.run()` API as a local agent, so it composes naturally with the rest of the framework.

In [15]:
from agent_framework.a2a import A2AAgent

remote_agent = A2AAgent(
    name="RemoteTravelAdvisor",
    url="http://localhost:9999/",
)

result = await remote_agent.run("What are the top 3 things to do in Kyoto?")
print(result)

In Kyoto, the top three things to do are:

1. **Kinkaku-ji (Golden Pavilion)**: This stunning Zen Buddhist temple, covered in gold leaf, is surrounded by beautiful gardens and reflects in the pond, creating a picturesque scene.

2. **Fushimi Inari Taisha**: Famous for its thousands of vermilion torii gates, this shrine dedicated to the rice deity Inari offers scenic hiking trails up Mount Inari, providing stunning views of the city.

3. **Arashiyama Bamboo Grove**: Walk through the serene and towering bamboo stalks of this iconic grove, and don’t miss the nearby Iwatayama Monkey Park for a unique experience with wild macaques.

Enjoy your trip to Kyoto!


In [16]:
# Multi-turn session works exactly like a local agent
session_r = remote_agent.create_session()

r1 = await remote_agent.run("I'm planning a 10-day trip to Japan. My budget is $4,000.", session=session_r)
print("Turn 1:", r1)

r2 = await remote_agent.run("Which cities should I prioritise given that budget?", session=session_r)
print("\nTurn 2:", r2)

Turn 1: That sounds like an exciting trip! Here’s a sample itinerary and budget breakdown for a 10-day visit to Japan within your $4,000 budget:

### Itinerary Suggestions:
**Days 1-3: Tokyo**
- Explore Shibuya, Shinjuku, and Akihabara.
- Visit Sensoji Temple in Asakusa.
- Spend a day at Tokyo Disneyland or DisneySea.

**Days 4-5: Kyoto**
- Take the Shinkansen (bullet train) to Kyoto (approx. $130).
- Visit Kinkaku-ji (Golden Pavilion) and Fushimi Inari Taisha.
- Experience a traditional tea ceremony.

**Days 6-7: Osaka**
- Take a short train ride to Osaka (approx. $15).
- Enjoy street food in Dotonbori.
- Visit Osaka Castle and Universal Studios Japan if interested.

**Days 8-9: Hiroshima**
- Travel to Hiroshima (approx. $100).
- Visit the Peace Memorial Park and Museum.
- Take a day trip to Miyajima Island.

**Day 10: Return to Tokyo**
- Head back to Tokyo for your flight departure (approx. $130).

### Budget Breakdown:
- **Flights (round-trip to Japan)**: $1,200 - $1,500
- **Accommo

### Streaming from a Remote Agent

If the server was started with `A2AExecutor(agent, stream=True)`, clients can request streaming.  
`run(stream=True)` returns an async generator of `AgentResponseUpdate` objects.

In [17]:
print("Streaming response: ", end="", flush=True)

async for update in await remote_agent.run("Give me a packing list for a beach holiday.", stream=True):
    text = getattr(update, 'text', None) or str(update)
    print(text, end="", flush=True)

print()  # newline

Streaming response: Sure! Here’s a packing list for a beach holiday:

**Clothing:**
1. Swimsuits (2-3)
2. Beach cover-up
3. Lightweight dresses or shorts
4. T-shirts or tank tops
5. Flip-flops or sandals
6. Sun hat or cap
7. Lightweight jacket or sweater (for cooler evenings)
8. Beach towel

**Beach Essentials:**
1. Sunscreen (SPF 30 or higher)
2. Sunglasses (UV protection)
3. Beach umbrella or sunshade
4. Beach bag
5. Reusable water bottle
6. Snacks or cooler for food

**Toiletries:**
1. After-sun lotion or aloe vera
2. Shampoo and conditioner 
3. Hairbrush or comb
4. Personal hygiene items

**Extras:**
1. Waterproof phone case
2. Book or e-reader
3. Beach games (frisbee, paddleball)
4. Snorkel gear (if applicable)
5. Camera or smartphone for photos

**Travel Documents:**
1. ID or passport
2. Travel insurance information
3. Reservations and itineraries

Pack according to your destination's weather, and don’t forget any personal items you might need! Enjoy your holiday!


### Remote Agent as a Tool

Because `A2AAgent` exposes `.as_tool()`, a local orchestrator can delegate to a remote agent  
the same way it delegates to any other agent — no protocol details leak into the orchestrator.

In [18]:
# Convert the remote agent into a tool
remote_travel_tool = remote_agent.as_tool(
    name="ask_travel_advisor",
    description="Ask the remote travel advisor any question about destinations, flights, or trip planning.",
    arg_name="question",
)

# Local orchestrator uses the remote agent as a tool — it has no idea it's remote
orchestrator = client.as_agent(
    name="trip_planner",
    instructions=(
        "You plan comprehensive trips for users. "
        "Use ask_travel_advisor to get travel information, then synthesise a clear plan."
    ),
    tools=[remote_travel_tool],
)

plan = await orchestrator.run("Plan a 7-day itinerary for a first-time visitor to Italy on a $2,500 budget.")
from IPython.display import display, Markdown
display(Markdown(str(plan)))

Here’s a comprehensive 7-day itinerary for a first-time visitor to Italy, tailored to a budget of $2,500:

### Day 1: Arrival in Rome
- **Accommodation**: Stay at a budget hostel or guesthouse like *The Beehive* or *Generator Rome*($30-50/night).
- **Activities**: 
  - Explore the historic center: Visit the Spanish Steps and Trevi Fountain (free).
- **Dining**: Have dinner at a local trattoria, such as *Trattoria Da Enzo* (pasta dishes around $10).

### Day 2: Rome
- **Activities**:
  - Visit Vatican City: Explore St. Peter's Basilica (free entry) and the Vatican Museums (entry around $17, reservations recommended).
- **Dining**: Grab a panino at *Pizzarium* or have some take-away pizza (~$5-10).

### Day 3: Florence
- **Travel**: Early train to Florence (approx. $30).
- **Accommodation**: Stay at *Plus Florence* with dormitory options ($30-50).
- **Activities**: 
  - Visit the Duomo for stunning views (entry around $10).
  - Explore the Uffizi Gallery (entry around $20).
- **Dining**: Dinner at *Trattoria Mario* for affordable local dishes (~$15).

### Day 4: Florence
- **Activities**: 
  - Day trip to Pisa (train round trip ~$20) to see the Leaning Tower, or explore the Tuscan countryside on a budget wine tour (~$60).
- **Dining**: Try local street food like *lampredotto* for a cheap lunch (~$5).

### Day 5: Venice
- **Travel**: Train to Venice (approx. $35).
- **Accommodation**: Stay in *Generator Venice* or a nearby hostel ($30-50).
- **Activities**: 
  - Stroll through St. Mark's Square.
  - Visit the Basilica (free entry) and explore the Rialto Market.
- **Dining**: Enjoy cicchetti (Venetian tapas) at a local bacaro (~$15-20).

### Day 6: Venice
- **Activities**: 
  - Take a Vaporetto (water bus) ride on the Grand Canal (single fare ~$7).
  - Consider visiting Murano for glass-making demonstrations (free).
- **Dining**:
  - Lunch from a local bakery (pizza or focaccia ~$5-8).
  - Dinner at *Osteria alle Testiere* for seafood (~$30, share plates for more value).

### Day 7: Departure
- **Travel**: Return to Rome by train (approx. $35) for your departure flight.
- **Last-minute Shopping**: Spend time in local shops for souvenirs before heading to the airport.

### Budget Breakdown:
- **Accommodation**: ~$300
- **Food**: ~$300 (averaging $3-15 per meal)
- **Transport**: ~$200 (trains and local transport)
- **Activities/Entry Fees**: ~$150-200
- **Miscellaneous**: ~$200 (souvenirs, extra dining, etc.)
- **Total**: ~$1,350 - $2,000 (leaving room for incidental expenses or a splurge!)

### Tips:
- **Booking**: Reserve transportation and accommodations in advance for the best rates.
- **Dining**: Use local markets for affordable meals.
- **Tours**: Consider “pay-what-you-wish” walking tours for an immersive experience.

This itinerary provides a blend of iconic sights and genuine local experiences, all while maintaining a budget-friendly approach. Enjoy your trip to Italy!

In [19]:
# Cleanup — terminate the background server
server_proc.terminate()
server_proc.wait()
print("Server stopped.")

Server stopped.


---
## Summary

| Feature | API |
|---|---|
| Custom memory | `ContextProvider` → `before_run` / `after_run` / `state` dict |
| Inject instructions | `context.extend_instructions(source_id, text)` |
| In-memory history | `InMemoryHistoryProvider(load_messages=True, ...)` |
| Persistent history | `FileHistoryProvider(storage_path=..., ...)` |
| Attach to agent | `agent.as_agent(..., context_providers=[provider])` |
| Expose agent as A2A service | `A2AExecutor(agent)` + `A2AStarletteApplication(card, handler).build()` |
| Connect to remote agent | `A2AAgent(name=..., url=...)` |
| Streaming from remote | `await remote.run(..., stream=True)` |
| Remote agent as tool | `remote.as_tool(name=..., arg_name=...)` |

The memory model follows a clean separation: **providers** own logic, **sessions** own state, and the framework calls providers in order on every `run()`.  
The A2A model follows the same principle: an `A2AAgent` is just an agent — it composes with everything else.